In [1]:
import os
import pandas as pd
import numpy as np
from pyproj import Transformer
from tqdm import tqdm

from IPython.display import display

In [3]:
ROOT_PATH = "dataset/scale_2_landscape/"

fp_lucas_train_val = os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train_val-0.06min.csv")
fp_lucas_train = os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train-0.06min.csv")
fp_lucas_val = os.path.join(ROOT_PATH, "lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_val-0.06min.csv")
fp_lucas_test = os.path.join(ROOT_PATH, "glc24_pa_test_private_CBN-med_matching-LUCAS-500m.csv")

df_lucas_train_val = pd.read_csv(fp_lucas_train_val)
df_lucas_test = pd.read_csv(fp_lucas_test)

print(f"LUCAS occurrences Train-Val: {df_lucas_train_val.shape} ({df_lucas_train_val['id'].nunique()} sites)")
display(df_lucas_train_val.head(1))
print(df_lucas_train_val.columns)
print(f"\nLUCAS occurrences Test: {df_lucas_test.shape} ({df_lucas_test['id'].nunique()} sites)")
display(df_lucas_test.head(1))
print(df_lucas_test.columns)

print("\nExcluding Test surveyIds from Train-val...")
df_lucas_train_val = df_lucas_train_val[~(df_lucas_train_val['id'].isin(df_lucas_test['id'].unique()))]
print(f"LUCAS occurrences Train-Val (after purge of Test): {df_lucas_test.shape} ({df_lucas_test['id'].nunique()} sites)")

relevant_cols_test = ['PlotObservationID_eva', 'surveyId', 'lon', 'lat', 'lucas_matching_ids']

LUCAS occurrences Train-Val: (47258, 12) (7903 sites)


,Unnamed: 0,id,file_path,full_path,image_source,exists,full_path_missing,full_path_2022,full_path_cover,lon,lat,subset
0,0,967805,2009/FR/378/823/37882398N.jpg,LUCAS/2009/FR/378/823/37882398N.jpg,file_path_gisco_north,False,LUCAS/../../lucas_missing/2009/FR/378/823/3788...,LUCAS/../../lucas_photos_all_2022/2009/FR/378/...,LUCAS/../../lucas_cover/2009/FR/378/823/378823...,3.303906,44.47791,train


Index(['Unnamed: 0', 'id', 'file_path', 'full_path', 'image_source', 'exists',
       'full_path_missing', 'full_path_2022', 'full_path_cover', 'lon', 'lat',
       'subset'],
      dtype='object')

LUCAS occurrences Test: (1252, 27) (64 sites)


,PlotObservationID_eva,lon,lat,year,datasetName,Access.regime,Expert.System,Cover.abundance.scale,geoUncertaintyInM,observer,...,region,surveyID,country,speciesId,surveyId,split,id,x_EPSG_32631,y_EPSG_32631,lucas_matching_ids
0,1953693.0,6.215064,43.12536,2021,CBNMed,2,V32,Braun/Blanquet (old),3.0,NaN,...,MEDITERRANEAN,93212,France,10822.0,74414,train,74414,761530.363056,4.779755e+06,751990


Index(['PlotObservationID_eva', 'lon', 'lat', 'year', 'datasetName',
       'Access.regime', 'Expert.System', 'Cover.abundance.scale',
       'geoUncertaintyInM', 'observer', 'areaInM2', 'coverTreeLayer',
       'coverShrubLayer', 'coverHerbLayer', 'coverMossLayer', 'source',
       'access', 'region', 'surveyID', 'country', 'speciesId', 'surveyId',
       'split', 'id', 'x_EPSG_32631', 'y_EPSG_32631', 'lucas_matching_ids'],
      dtype='object')

Excluding Test surveyIds from Train-val...
LUCAS occurrences Train-Val (after purge of Test): (1252, 27) (64 sites)


In [11]:
def delta_after_shift(d, lon, lat):
    """Computes the delta between 2 WGS84 coords based on a metric distance, through the EPSG:3035.
    
    WARNING: extremely slow to call within a loop.
    
    d: distance in meters (added to both x and y in EPSG:3035)
    lon, lat: original WGS84 coordinates (degrees)

    Returns (delta_lon, delta_lat) between before and after.
    """
    wgs84 = "EPSG:4326"    # lon/lat in WGS84
    epsg3035 = "EPSG:3035" # ETRS89 / LAEA Europe

    # always_xy=True → inputs/outputs are (lon, lat) for geographic CRS
    to_3035 = Transformer.from_crs(wgs84, epsg3035, always_xy=True)
    to_4326 = Transformer.from_crs(epsg3035, wgs84, always_xy=True)

    # 1) WGS84 → EPSG:3035
    x0, y0 = to_3035.transform(lon, lat)

    # 2) Add distance d to each projected coordinate
    x1 = x0 + d
    y1 = y0 + d

    # 3) Back to WGS84
    lon1, lat1 = to_4326.transform(x1, y1)

    # 4) Delta between before and after
    dlon = lon1 - lon
    dlat = lat1 - lat

    return dlon, dlat

def apply_noise(df: pd.DataFrame,
                columns: list[str],
                gamma: float = 0.1,
                epsilon: float = 0.5,
                noise_type: str = "uniform",
                random_state: int = None
               ):
    """
    Apply random noise to a pandas column.

    Args:
        df (pd.DataFrame): Input dataframe.
        column (str): Column name to modify.
        gamma (float): Probability of applying noise to each value (0 to 1).
        epsilon (float): Noise magnitude.
        noise_type (str): Type of noise to apply ("gaussian" or "uniform").
        random_state (int, optional): Seed for reproducibility.

    Returns:
        pd.DataFrame: New dataframe with the noisy column.
    """
    if random_state is not None:
        np.random.seed(random_state)

    df_noisy = df.copy()

    # Decide which rows get noise
    mask = np.random.rand(len(df)) < gamma

    # Generate noise
    if noise_type == "gaussian":
        noise = np.random.normal(loc=0, scale=epsilon, size=len(df))
    elif noise_type == "uniform":
        noise = np.random.uniform(low=epsilon, high=5*epsilon, size=len(df))
    else:
        raise ValueError("noise_type must be 'gaussian' or 'uniform'")
    signs = np.random.choice([-1, 1], size=len(df))

    # Apply noise only where mask is True
    for column in columns:
        df_noisy.loc[mask, column] += signs[mask] * noise[mask]
        df_noisy[f'{column}_original'] = df[column]

    return df_noisy

def apply_noise_over_GPS_in_meters(
    df: pd.DataFrame,
    gamma: float = 0.1,
    epsilon: int = 100,
    random_state: int = None
):
    """
    Apply random noise to a pandas column.

    Args:
        df (pd.DataFrame): Input dataframe.
        column (str): Column name to modify.
        gamma (float): Probability of applying noise to each value (0 to 1).
        epsilon (float): Noise magnitude.
        noise_type (str): Type of noise to apply ("gaussian" or "uniform").
        random_state (int, optional): Seed for reproducibility.

    Returns:
        pd.DataFrame: New dataframe with the noisy column.
    """
    if random_state is not None:
        np.random.seed(random_state)

    df_noisy = df.copy()
    
    wgs84 = "EPSG:4326"    # lon/lat in WGS84
    epsg3035 = "EPSG:3035" # ETRS89 / LAEA Europe
    # always_xy=True → inputs/outputs are (lon, lat) for geographic CRS
    to_3035 = Transformer.from_crs(wgs84, epsg3035, always_xy=True)
    to_4326 = Transformer.from_crs(epsg3035, wgs84, always_xy=True)

    # Decide which rows get noise
    mask = np.random.rand(len(df)) < gamma
    signs_lon = np.random.choice([-1, 1], size=len(df))
    signs_lat = np.random.choice([-1, 1], size=len(df))

    # Apply noise only where mask is True
    df_noisy['lon_noisy'] = df['lon'].copy()
    df_noisy['lat_noisy'] = df['lat'].copy()
    df_noisy['gps_match'] = [True]*len(df)
    for (rowi, row), maski in tqdm(zip(df.iterrows(), mask), total=(len(df))):
        if maski:
            lon = row['lon']
            lat = row['lat']
            x0, y0 = to_3035.transform(lon, lat)
            # 2) Add distance d to each projected coordinate
            x1 = x0 + epsilon
            y1 = y0 + epsilon
            # 3) Back to WGS84
            lon1, lat1 = to_4326.transform(x1, y1)
            # 4) Delta between before and after
            d_lon = lon1 - lon
            d_lat = lat1 - lat

            df_noisy.loc[rowi, 'lon_noisy'] += signs_lon[rowi] * d_lon
            df_noisy.loc[rowi, 'lat_noisy'] += signs_lat[rowi] * d_lat
            df_noisy.loc[rowi, 'gps_match'] = False
       
    return df_noisy

gps_error_probability = 0.25
gps_error_in_meters = 1000

In [12]:
df_lucas_train_val_noisy_gps = apply_noise_over_GPS_in_meters(df_lucas_train_val, gamma=gps_error_probability, epsilon=gps_error_in_meters)

print(df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['lon'] != df_lucas_train_val_noisy_gps['lon_noisy']].shape)
print(df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['lon'] == df_lucas_train_val_noisy_gps['lon_noisy']].shape)
display(df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['lon'] != df_lucas_train_val_noisy_gps['lon_noisy']].head(3)[['lon', 'lon_noisy', 'gps_match']])
display(df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['lon'] == df_lucas_train_val_noisy_gps['lon_noisy']].head(3)[['lon', 'lon_noisy', 'gps_match']])

100%|██████████████████████████████████| 47258/47258 [00:03<00:00, 14336.40it/s]

(11709, 15)
(35549, 15)


,lon,lon_noisy,gps_match
1,3.019830,3.008649,False
9,4.019160,4.007704,False
11,6.088003,6.099651,False


,lon,lon_noisy,gps_match
0,3.303906,3.303906,True
2,5.501157,5.501157,True
3,3.273118,3.273118,True


In [13]:
df_lucas_train_val_noisy_gps.to_csv(f"{fp_lucas_train_val.split('.csv')[0]}_noisy_100m.csv", index=False)
df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['subset'] == 'train'].to_csv(f"{fp_lucas_train.split('.csv')[0]}_noisy_100m.csv", index=False)
df_lucas_train_val_noisy_gps[df_lucas_train_val_noisy_gps['subset'] == 'val'].to_csv(f"{fp_lucas_val.split('.csv')[0]}_noisy_100m.csv", index=False)

In [14]:
df_lucas_test_unique_sId = df_lucas_test[relevant_cols_test].drop_duplicates(subset='surveyId', keep="first", ignore_index=True)
df_lucas_test_unique_sId['id'] = df_lucas_test_unique_sId['surveyId']

df_lucas_test_noisy_gps = apply_noise_over_GPS_in_meters(df_lucas_test_unique_sId, gamma=gps_error_probability, epsilon=gps_error_in_meters)
display(df_lucas_test_noisy_gps[df_lucas_test_noisy_gps['lon'] != df_lucas_test_noisy_gps['lon_noisy']].sample(3))

100%|████████████████████████████████████████| 64/64 [00:00<00:00, 13635.86it/s]


,PlotObservationID_eva,surveyId,lon,lat,lucas_matching_ids,id,lon_noisy,lat_noisy,gps_match
30,1951972.0,1383354,3.787923,43.49644,1021041 1016806 1011135,1383354,3.799265,43.486706,False
1,1949970.0,193349,6.182597,43.14030,668151 667758 661532 484357,193349,6.194260,43.130829,False
57,1951976.0,3380395,3.788352,43.49606,1021041 1016806 1011135,3380395,3.799694,43.486326,False


In [15]:
df_lucas_test_noisy_gps.to_csv(f"{fp_lucas_test.split('.csv')[0]}_noisy_{gps_error_in_meters}m.csv", index=False)

## Insert file paths in test set

In [16]:
fp_lucas_train_val_noisy = os.path.join(ROOT_PATH, 'lucas_harmo_cover_exif_nona_fixed_gps_CBN-Med_expanded_essentials_exists_train_val-0.06min_noisy_100m.csv')
fp_lucas_test_noisy =  os.path.join(ROOT_PATH, 'glc24_pa_test_private_CBN-med_matching-LUCAS-500m_noisy_1000m.csv')

df_train_val_noisy = pd.read_csv(fp_lucas_train_val_noisy)
df_test_noisy = pd.read_csv(fp_lucas_test_noisy)

print(df_train_val_noisy.columns)
print(df_test_noisy.columns)
print(df_test_noisy.sample(1)['lucas_matching_ids'])

Index(['Unnamed: 0', 'id', 'file_path', 'full_path', 'image_source', 'exists',
       'full_path_missing', 'full_path_2022', 'full_path_cover', 'lon', 'lat',
       'subset', 'lon_noisy', 'lat_noisy', 'gps_match'],
      dtype='object')
Index(['PlotObservationID_eva', 'surveyId', 'lon', 'lat', 'lucas_matching_ids',
       'id', 'lon_noisy', 'lat_noisy', 'gps_match'],
      dtype='object')
51    1095053 1086393 1081666 
Name: lucas_matching_ids, dtype: object


In [17]:
df_test_noisy['file_path'] = ['']*len(df_test_noisy)
for rowi, row in tqdm(df_test_noisy.copy().iterrows(), total=df_test_noisy.copy().shape[0]):
    fps_lucas = ''
    lucas_ids = row['lucas_matching_ids'].split()
    for k, lucas_id in enumerate(lucas_ids):
        df_slice = df_train_val_noisy[df_train_val_noisy['id'] == int(lucas_id)]
        file_paths = df_slice['file_path'].tolist()
        if file_paths == []:
            print(f"[WARNING] Lucas ID {df_slice['id']} not found")
        fps_lucas += ' '.join(file_paths)
        fps_lucas += ';' if len(lucas_ids) > 1 and k < len(lucas_ids)-1 else ''
    df_test_noisy.loc[rowi, 'lucas_matching_ids'] = ';'.join(lucas_ids)
    df_test_noisy.loc[rowi, 'file_path'] = fps_lucas

100%|█████████████████████████████████████████| 64/64 [00:00<00:00, 2073.71it/s]


In [18]:
display(df_test_noisy.sample(1)[['lon', 'lat', 'id', 'lucas_matching_ids', 'file_path']])

,lon,lat,id,lucas_matching_ids,file_path
39,5.19553,43.82637,2049889,679793,2012/FR/393/423/39342314N.jpg 2012/FR/393/423/...


In [19]:
df_test_noisy_surveyIds_groupby = pd.DataFrame({}, columns=relevant_cols_test)
display(df_test_noisy_surveyIds_groupby)

df_test_noisy_surveyIds_groupby = df_test_noisy.drop_duplicates(subset="surveyId", keep="first")
display(df_test_noisy_surveyIds_groupby)

,PlotObservationID_eva,surveyId,lon,lat,lucas_matching_ids


,PlotObservationID_eva,surveyId,lon,lat,lucas_matching_ids,id,lon_noisy,lat_noisy,gps_match,file_path
0,1953693.0,74414,6.215064,43.125360,751990,74414,6.215064,43.125360,True,2012/FR/401/222/40122232N.jpg 2012/FR/401/222/...
1,1949970.0,193349,6.182597,43.140300,668151;667758;661532;484357,193349,6.194260,43.130829,False,2009/FR/401/022/40102234N.jpg 2009/FR/401/022/...
2,1952624.0,283744,5.152670,43.750800,1095053;1086393;1081666,283744,5.152670,43.750800,True,2009/FR/393/023/39302306N.jpg 2009/FR/393/023/...
3,1951977.0,414289,3.788459,43.495870,1021041;1016806;1011135,414289,3.788459,43.495870,True,2009/FR/381/822/38182286N.jpg 2009/FR/381/822/...
4,1951979.0,458106,3.788613,43.495630,1021041;1016806;1011135,458106,3.788613,43.495630,True,2009/FR/381/822/38182286N.jpg 2009/FR/381/822/...
...,...,...,...,...,...,...,...,...,...,...
59,1952165.0,3598701,3.807560,43.569020,369603,3598701,3.807560,43.569020,True,2018/FR/382/022/38202294N.jpg 2018/FR/382/022/...
60,1952627.0,3722243,5.150750,43.751700,1095053;1086393;1081666,3722243,5.150750,43.751700,True,2009/FR/393/023/39302306N.jpg 2009/FR/393/023/...
61,1953371.0,3744305,6.572457,43.268870,1040360;1034859;1027039,3744305,6.572457,43.268870,True,2009/FR/404/222/40422246N.jpg 2009/FR/404/222/...
62,1952161.0,3765275,3.229080,43.427344,914102;913730;907186;519111,3765275,3.229080,43.427344,True,2009/FR/377/222/37722282N.jpg 2009/FR/377/222/...


In [20]:
df_test_noisy.to_csv(fp_lucas_test_noisy, index=False)

In [21]:
df_test_noisy[df_test_noisy['id']==574217]['file_path'].sample(1).values

array(['2009/FR/377/222/37722282N.jpg 2009/FR/377/222/37722282S.jpg 2009/FR/377/222/37722282E.jpg 2009/FR/377/222/37722282W.jpg 2009/FR/377/222/37722282P.jpg 2009/FR/377/222/37722282C.jpg;2012/FR/377/222/37722282N.jpg 2012/FR/377/222/37722282S.jpg 2012/FR/377/222/37722282E.jpg 2012/FR/377/222/37722282W.jpg 2012/FR/377/222/37722282P.jpg 2012/FR/377/222/37722282C.jpg;2015/FR/377/222/37722282N.jpg 2015/FR/377/222/37722282S.jpg 2015/FR/377/222/37722282E.jpg 2015/FR/377/222/37722282W.jpg 2015/FR/377/222/37722282P.jpg 2015/FR/377/222/37722282C.jpg;2006/FR/377/222/37722282N.jpg 2006/FR/377/222/37722282S.jpg 2006/FR/377/222/37722282E.jpg 2006/FR/377/222/37722282W.jpg 2006/FR/377/222/37722282P.jpg 2006/FR/377/222/37722282C.jpg'],
      dtype=object)